<a href="https://colab.research.google.com/github/shahabazkc/Whisper-finetuned-quran-dataset/blob/main/Whisper_large_with_finetuned_quran_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torchaudio librosa

## Local Inference on GPU
Model page: https://huggingface.co/tarteel-ai/whisper-base-ar-quran

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/tarteel-ai/whisper-base-ar-quran)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"

# processor = AutoProcessor.from_pretrained("tarteel-ai/whisper-base-ar-quran")
# model = AutoModelForSpeechSeq2Seq.from_pretrained(
#     "tarteel-ai/whisper-base-ar-quran"
# ).to(device)

MODEL_NAME = "shahabazkc10/whisper-large-v3-turbo-ar-quran-mix-norm"
# other options:
# "shahabazkc10/whisper-medium-ar-quran-mix-norm"
# "shahabazkc10/whisper-small-ar-quran-mix-norm"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

Saving ar rahman.wav to ar rahman.wav


In [ ]:
import librosa
import numpy as np

audio, sr = librosa.load(audio_path, sr=16000)

chunk_length = 20 * sr  # 20 sec chunks (better for Qur’an)

chunks = [
    audio[i:i + chunk_length]
    for i in range(0, len(audio), chunk_length)
]

In [ ]:
full_transcription = ""

for chunk in chunks:
    inputs = processor(
        chunk,
        sampling_rate=16000,
        return_tensors="pt"
    )

    # dtype fix for large model
    inputs = {k: v.to(device).to(model.dtype) for k, v in inputs.items()}

    with torch.no_grad():
        predicted_ids = model.generate(
            inputs["input_features"],
            max_length=256,
            num_beams=3,
            temperature=0.0
        )

    text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    print(text)
    full_transcription += text + " "

print("\nFinal Output:\n", full_transcription)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

الرحمن علم القران
خلق الانسان علمه البيان الشمس والقمر بحسبان والنجم والشجر يسجدان
والسما رفعها ووضع الميزان الا تتغوا في الميزان واقيموا الوزن بالقسط ولا تخسروا الميزان
والارض وضعها للانام فيها فاكهة والنخل ذات الاكمام والحب ذو العصف والريحان فباي الاق
الله ربكما تكذبان خلك الانسان من سلصال كالفخار وخلك الجان من مارج
والنار فباي الا ربكما تكذبان رب المشرقين ورب المغربين فباي الا
ربكما تكذبان

Final Output:
 الرحمن علم القران خلق الانسان علمه البيان الشمس والقمر بحسبان والنجم والشجر يسجدان والسما رفعها ووضع الميزان الا تتغوا في الميزان واقيموا الوزن بالقسط ولا تخسروا الميزان والارض وضعها للانام فيها فاكهة والنخل ذات الاكمام والحب ذو العصف والريحان فباي الاق الله ربكما تكذبان خلك الانسان من سلصال كالفخار وخلك الجان من مارج والنار فباي الا ربكما تكذبان رب المشرقين ورب المغربين فباي الا ربكما تكذبان 
